In [ ]:
# Imports

import tensorflow as tf
from tensorflow import keras
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import kagglehub
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, recall_score
from sklearn.impute import SimpleImputer

In [ ]:
# Load Fashion MNIST dataset
(X_train, y_train) , (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

In this assignment, you are supposed to classify the Fashion MNIST dataset 
items using a neural network-based model from the Tensorflow Keras library. 

In [ ]:
# Check the shape of the data

print(X_train.shape, X_test.shape)


# RGB values are from 0-255, we scale them to be between 0 and 1 (Grayscale)
X_train = (X_train / 255)
X_test = (X_test / 255)

# Creates flattened versions of the images for dense exlusive model
X_train_flattened = X_train.reshape(len(X_train), 28*28)
X_test_flattened = X_test.reshape(len(X_test), 28*28)

In [ ]:
# Shows the first 25 images in the training set

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    img = X_train[i]
    # remove channel dim if present (python is stupid)
    if img.ndim == 3 and img.shape[2] == 1:
        img = img.squeeze()
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
plt.show()

In [ ]:
# Build a simple dense model

modelNN = keras.models.Sequential()
modelNN.add(keras.layers.InputLayer(input_shape=(784,)))
modelNN.add(keras.layers.Dense(10, activation='softmax'))


# Build a simple CNN model

# ReLU activation function is used for hidden layers, it converts negative values to zero
# Softmax activation function allows us to limit the output to a probability distribution over the classes

modelCNN = keras.models.Sequential()
modelCNN.add(keras.layers.InputLayer(input_shape=(28,28,1)))
modelCNN.add(keras.layers.Conv2D(28, kernel_size=(3,3), activation='relu'))
modelCNN.add(keras.layers.MaxPooling2D())
modelCNN.add(keras.layers.Conv2D(56, kernel_size=(3,3), activation='relu'))
modelCNN.add(keras.layers.MaxPooling2D())
modelCNN.add(keras.layers.Conv2D(112, kernel_size=(3,3), activation='relu'))
modelCNN.add(keras.layers.Flatten())
modelCNN.add(keras.layers.Dense(64, activation='relu'))
modelCNN.add(keras.layers.Dense(10, activation='softmax'))


In [ ]:
# Summarize the modelNN
modelNN.summary()

In [ ]:
# Summarize the modelCNN
modelCNN.summary()

In [ ]:
# Adam combines AdaGrad and RMSProp algorithms for efficient training
# AdaGrad and RMSProp are already recommended for MNIST datasets, and Adam combines their advantages
# SparseCategoricalCrossentropy computes the cross-entropy loss between the labels and predictions when the labels are provided as integers
# Cross-entropy is a measure of the difference between two probability distributions, in this case the true labels and predicted labels

# Train the modelNN

modelNN.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])


# Train the modelCNN

modelCNN.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])


In [ ]:
# Using 10% of the training data for validation
# Does 10 epochs (iterations over the entire dataset)

# Fit the modelNN

modelNN.fit(X_train_flattened, y_train,validation_split=0.1,epochs=10)


# Fit the modelCNN

modelCNN.fit(X_train, y_train,validation_split=0.1,epochs=10)

In [ ]:
# Make predictions with both models
y_predicted_NN = modelNN.predict(X_test_flattened)
y_predicted_CNN = modelCNN.predict(X_test)

In [ ]:
# Get the predicted labels by finding the index of the maximum value in each prediction array
y_predicted_NN_labels = [np.argmax(i) for i in y_predicted_NN]
y_predicted_CNN_labels = [np.argmax(i) for i in y_predicted_CNN]

In [ ]:
# Confusion matrix  for both models
cmNN = tf.math.confusion_matrix(labels=y_test,predictions=y_predicted_NN_labels)
cmCNN = tf.math.confusion_matrix(labels=y_test,predictions=y_predicted_CNN_labels)

In [ ]:
# Plot confusion matrix for both models

import seaborn as sn
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

class_names = [
    "0 T-shirt/top", "1 Trouser", "2 Pullover", "3 Dress", "4 Coat",
    "5 Sandal", "6 Shirt", "7 Sneaker", "8 Bag", "9 Ankle boot"
]

cm_nn = cmNN.numpy() if hasattr(cmNN, "numpy") else np.array(cmNN)
cm_cnn = cmCNN.numpy() if hasattr(cmCNN, "numpy") else np.array(cmCNN)
vmax = max(cm_nn.max(), cm_cnn.max())

sn.heatmap(cm_nn, annot=True, fmt='d',
           xticklabels=class_names, yticklabels=class_names,
           cmap='Blues', ax=axes[0], vmin=0, vmax=vmax)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title('NN Confusion Matrix')

sn.heatmap(cm_cnn, annot=True, fmt='d',
           xticklabels=class_names, yticklabels=class_names,
           cmap='Blues', ax=axes[1], vmin=0, vmax=vmax)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('CNN Confusion Matrix')

plt.tight_layout()
plt.show()